# Ollama en local

Cuaderno para instalar, verificar y consumir Ollama localmente desde Windows. No requiere API keys ni créditos.

## 1. Verificar entorno local y dependencias

Ejecuta esta celda para conocer el intérprete, el sistema y si el puerto habitual de Ollama está ocupado.

In [13]:
import platform
import shutil
import socket
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"Sistema: {platform.platform()}")
print(f"Ollama CLI: {shutil.which('ollama') or 'no encontrado'}")
print(f"curl: {shutil.which('curl') or 'no encontrado'}")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
    port_in_use = connection.connect_ex(("127.0.0.1", 11434)) == 0
print(f"Puerto 11434 ocupado: {port_in_use}")

Python: 3.12.10
Sistema: Windows-11-10.0.26200-SP0
Ollama CLI: C:\Users\oitav\AppData\Local\Programs\Ollama\ollama.EXE
curl: C:\Windows\system32\curl.EXE
Puerto 11434 ocupado: True


## 2. Instalar y validar Ollama CLI

Ejecuta la instalación una sola vez en PowerShell. Después reinicia la terminal o VS Code.

In [14]:
# Ejecuta en PowerShell, fuera del notebook, si Ollama no está instalado:
# winget install Ollama.Ollama

import subprocess

try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, check=True)
    print(result.stdout.strip() or result.stderr.strip())
except FileNotFoundError:
    print("Ollama no está instalado. Ejecuta: winget install Ollama.Ollama")
except subprocess.CalledProcessError as error:
    print(error.stderr.strip())

ollama version is 0.32.15


## 3. Iniciar el servicio de Ollama y comprobar estado

La aplicación de Ollama suele iniciar el servicio automáticamente. Si no, ejecuta `ollama serve` en una terminal aparte.

In [15]:
import requests

BASE_URL = "http://127.0.0.1:11434"

try:
    response = requests.get(f"{BASE_URL}/api/tags", timeout=3)
    response.raise_for_status()
    installed_models = [model["name"] for model in response.json().get("models", [])]
    print("Servicio disponible. Modelos instalados:")
    print("\n".join(installed_models) or "Aún no hay modelos descargados.")
except requests.RequestException as error:
    raise RuntimeError("No se puede conectar con Ollama. Abre Ollama o ejecuta 'ollama serve'.") from error

Servicio disponible. Modelos instalados:
qwen2.5:3b
llama3.2:1b


## 4. Descargar un modelo para uso local

`qwen2.5:3b` es ligero y adecuado para probar en CPU. La descarga se realiza una sola vez.

In [16]:
MODEL = "qwen2.5:3b"

# Descarga el modelo. Puede tardar varios minutos según la conexión.
result = subprocess.run(["ollama", "pull", MODEL], text=True)
if result.returncode == 0:
    print(f"Modelo listo: {MODEL}")
    subprocess.run(["ollama", "list"], text=True)

Modelo listo: qwen2.5:3b


## 5. Ejecutar prompts desde la terminal

Usa esta celda para hacer una prueba simple de inferencia local mediante el CLI.

In [17]:
import requests

MODEL = "qwen2.5:3b"
BASE_URL = "http://127.0.0.1:11434"
prompt = "Explica en dos frases qué es un modelo de lenguaje local."

print(f"Consultando Ollama con {MODEL}...")
response = requests.post(
    f"{BASE_URL}/api/generate",
    json={
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2, "num_predict": 80},
    },
    timeout=120,
)
response.raise_for_status()
print(response.json()["response"].strip())

Consultando Ollama con qwen2.5:3b...
Un modelo de lenguaje local es una versión personalizada de un modelo de lenguaje preexistente que ha sido entrenado con datos específicos de una región geográfica o comunidad lingüística, lo que le permite entender y generar texto de manera más precisa y relevante para ese contexto particular.


## 6. Consumir la API local de Ollama con Python

Estas funciones usan solo `http://127.0.0.1:11434`, sin enviar datos fuera del equipo.

In [18]:
SYSTEM_PROMPT = "Eres un asistente útil. Responde siempre en español y de forma concisa."
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

def generate(prompt: str, options: dict | None = None) -> str:
    payload = {"model": MODEL, "prompt": prompt, "stream": False, "options": options or {}}
    response = requests.post(f"{BASE_URL}/api/generate", json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["response"].strip()

def chat(user_message: str) -> str:
    messages.append({"role": "user", "content": user_message})
    response = requests.post(
        f"{BASE_URL}/api/chat",
        json={"model": MODEL, "messages": messages, "stream": False},
        timeout=120,
    )
    response.raise_for_status()
    answer = response.json()["message"]["content"].strip()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(generate("Di hola en español."))
print(chat("¿Qué puedes hacer de forma local?"))

¡Hola! ¿Cómo estás?
Puedo responder consultas de forma local, como proporcionar información sobre lugares cercanos si se me proporciona su ubicación. También puedo asistir con tareas locales como organizar fechas de eventos o servicios en su área.


## 7. Implementar respuestas en streaming

La respuesta aparece por fragmentos mientras se genera.

In [19]:
import json


def stream_generate(prompt: str, options: dict | None = None) -> str:
    payload = {"model": MODEL, "prompt": prompt, "stream": True, "options": options or {}}
    response = requests.post(f"{BASE_URL}/api/generate", json=payload, stream=True, timeout=120)
    response.raise_for_status()
    chunks = []
    for line in response.iter_lines(decode_unicode=True):
        if line:
            chunk = json.loads(line).get("response", "")
            chunks.append(chunk)
            print(chunk, end="", flush=True)
    print()
    return "".join(chunks)


stream_generate("Describe una ventaja de ejecutar un LLM en local.")

Ejecutar un LLM (Lenguaje de Modelo de Lenguaje, también conocido como LLM) en local tiene varias ventajas, especialmente en términos de control, privacidad y rendimiento. Aquí te presento una de las principales ventajas:

**Mejor control de la privacidad y seguridad:** 

Cuando los LLMs se ejecutan en un entorno local, puedes tener un mayor control sobre la privacidad y seguridad de los datos. Puedes asegurarte de que solo los datos que deseas se procesan en tu entorno local, protegiendo así los datos sensibles o confidenciales de tus usuarios. Si los datos son enviados a un LLM en la nube, puedes no controlar completamente su uso y almacenamiento.

Por ejemplo, si tu aplicación necesita realizar procesos de aprendizaje de máquina o análisis de texto que requieren un gran volumen de datos, y quieres asegurarte de que estos datos no se expidan a un tercero, ejecutar el LLM en tu propio entorno local permite que todos los datos estén bajo tu control y protección.


'Ejecutar un LLM (Lenguaje de Modelo de Lenguaje, también conocido como LLM) en local tiene varias ventajas, especialmente en términos de control, privacidad y rendimiento. Aquí te presento una de las principales ventajas:\n\n**Mejor control de la privacidad y seguridad:** \n\nCuando los LLMs se ejecutan en un entorno local, puedes tener un mayor control sobre la privacidad y seguridad de los datos. Puedes asegurarte de que solo los datos que deseas se procesan en tu entorno local, protegiendo así los datos sensibles o confidenciales de tus usuarios. Si los datos son enviados a un LLM en la nube, puedes no controlar completamente su uso y almacenamiento.\n\nPor ejemplo, si tu aplicación necesita realizar procesos de aprendizaje de máquina o análisis de texto que requieren un gran volumen de datos, y quieres asegurarte de que estos datos no se expidan a un tercero, ejecutar el LLM en tu propio entorno local permite que todos los datos estén bajo tu control y protección.'

## 8. Configurar parámetros del modelo y plantillas

Ajusta creatividad, longitud máxima y el prompt de sistema para comparar el comportamiento.

In [20]:
creative_options = {"temperature": 0.9, "top_p": 0.95, "num_predict": 120}
precise_options = {"temperature": 0.2, "top_p": 0.7, "num_predict": 60}
prompt = "Propón un nombre para un agente local de IA."

print("Creativo:", generate(prompt, creative_options))
print("\nPreciso:", generate(prompt, precise_options))

Creativo: Cómo el Agente Líder IA Local

Preciso: Un nombre creativo para un agente local de IA podría ser "LocAI". Este nombre combina la palabra "local" con la inicial de "Inteligencia Artificial", lo que refleja su naturaleza localizada y su función de inteligencia artificial.


## 9. Manejar errores, timeouts y logs

Esta función devuelve errores de conexión o de modelo de forma legible y permite reintentos simples.

In [21]:
import time

def safe_generate(prompt: str, attempts: int = 2) -> str | None:
    for attempt in range(1, attempts + 1):
        try:
            return generate(prompt)
        except requests.Timeout:
            print(f"Tiempo de espera agotado (intento {attempt}/{attempts}).")
        except requests.HTTPError as error:
            print(f"Error HTTP {error.response.status_code}: {error.response.text}")
            return None
        except requests.RequestException as error:
            print(f"Error de conexión: {error}")
        time.sleep(1)
    return None

safe_generate("Responde solamente: servicio local operativo.")

# Para ver mensajes del servicio desde PowerShell: ollama serve

'servicio local operativo.'

## 10. Crear pruebas rápidas de latencia y uso básico

Mide varias peticiones cortas. La tabla incluye tiempo de respuesta y longitud aproximada de la salida.

In [22]:
import time

import pandas as pd
import requests

MODEL = "qwen2.5:3b"
BASE_URL = "http://127.0.0.1:11434"


def generate(prompt: str, options: dict | None = None) -> str:
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": options or {},
    }
    response = requests.post(
        f"{BASE_URL}/api/generate",
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["response"].strip()


benchmark_prompts = [
    "Di hola.",
    "¿Cuánto es 7 por 8?",
    "Define IA local en una frase.",
]
results = []

for benchmark_prompt in benchmark_prompts:
    started_at = time.perf_counter()
    answer = generate(benchmark_prompt, {"temperature": 0, "num_predict": 40})
    elapsed_seconds = time.perf_counter() - started_at
    results.append(
        {
            "prompt": benchmark_prompt,
            "segundos": round(elapsed_seconds, 2),
            "caracteres": len(answer),
            "respuesta": answer,
        }
    )

pd.DataFrame(results)

,prompt,segundos,caracteres,respuesta
0,Di hola.,1.56,47,¡Hola! ¿Cómo estás? ¿En qué puedo ayudarte hoy?
1,¿Cuánto es 7 por 8?,0.91,14,7 por 8 es 56.
2,Define IA local en una frase.,2.87,177,IA local se refiere a la implementación de int...


## 11. Construir un agente de voz con librerias open source

Esta sección adapta el modelo de `Version_online_agente_voz.ipynb` a una arquitectura local. Sustituye OpenAI STT por `faster-whisper`, mantiene Silero VAD, utiliza Ollama como LLM y propone Piper como TTS local.

```text
Microfono -> VAD -> faster-whisper -> Ollama -> Piper -> Altavoces
```

El transporte puede ser audio local con `sounddevice` o LiveKit autohospedado. Primero se prueba el flujo por turnos y después se incorpora el micrófono en tiempo real.

In [23]:
import importlib.util
import shutil

import requests

BASE_URL = "http://127.0.0.1:11434"
open_source_modules = {
    "faster-whisper": importlib.util.find_spec("faster_whisper") is not None,
    "silero-vad": importlib.util.find_spec("silero_vad") is not None,
    "sounddevice": importlib.util.find_spec("sounddevice") is not None,
    "soundfile": importlib.util.find_spec("soundfile") is not None,
}

print(f"Ollama CLI: {shutil.which('ollama') or 'no encontrado'}")
print("Dependencias open source:")
for package_name, available in open_source_modules.items():
    print(f"- {package_name}: {'disponible' if available else 'no instalado'}")

try:
    response = requests.get(f"{BASE_URL}/api/tags", timeout=3)
    response.raise_for_status()
    model_names = [model["name"] for model in response.json().get("models", [])]
    print(f"Ollama disponible: sí ({len(model_names)} modelo(s))")
    print("Modelos:", ", ".join(model_names) or "ninguno")
except requests.RequestException as error:
    print(f"Ollama disponible: no ({error})")

Ollama CLI: C:\Users\oitav\AppData\Local\Programs\Ollama\ollama.EXE
Dependencias open source:
- faster-whisper: disponible
- silero-vad: disponible
- sounddevice: disponible
- soundfile: disponible
Ollama disponible: sí (2 modelo(s))
Modelos: qwen2.5:3b, llama3.2:1b


## 12. Instalar dependencias de voz open source

El entorno ya dispone de `faster-whisper` y `sounddevice`. Para completar el flujo local, instala `silero-vad` y `soundfile` desde una terminal con el entorno `voiceagent` activo:

```powershell
python -m pip install silero-vad soundfile
```

Piper requiere además instalar su ejecutable y descargar una voz compatible. La voz y sus pesos pueden tener una licencia diferente a la de la librería, así que hay que revisarla antes de distribuir el proyecto.

In [ ]:
import subprocess
import sys
from pathlib import Path


class PiperSynthesizer:
    def __init__(
        self,
        voice_name: str = "es_ES-davefx-medium",
        data_dir: str = "../models/piper",
    ) -> None:
        self.voice_name = voice_name
        self.data_dir = Path(data_dir).resolve()

    def synthesize(self, text: str, output_path: str) -> str:
        output_file = Path(output_path).resolve()
        output_file.parent.mkdir(parents=True, exist_ok=True)
        self.data_dir.mkdir(parents=True, exist_ok=True)
        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "piper",
                "--model",
                self.voice_name,
                "--data-dir",
                str(self.data_dir),
                "--output_file",
                str(output_file),
            ],
            input=text + "\n",
            text=True,
            encoding="utf-8",
            capture_output=True,
        )
        if result.returncode != 0:
            details = result.stderr.strip() or result.stdout.strip()
            raise RuntimeError(f"Piper no pudo generar el audio: {details}")
        return str(output_file)

piper_synthesizer = PiperSynthesizer()
print("PiperSynthesizer preparado con la voz es_ES-davefx-medium.")

Un agente de voz local es una tecnología que permite la reproducción de voz en tiempo real en dispositivos físicos.


## 13. Transcripción local con faster-whisper

`faster-whisper` sustituye el STT de OpenAI. Descarga un modelo la primera vez que se crea el transcriptor; el tamaño `base` es un punto de partida razonable para CPU.

In [ ]:
from pathlib import Path

import sounddevice as sd
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE = 16_000
RECORD_SECONDS = 5
AUDIO_DIR = Path("../models/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)


def record_microphone(
    output_path: str = str(AUDIO_DIR / "usuario.wav"),
    duration: int = RECORD_SECONDS,
    sample_rate: int = SAMPLE_RATE,
) -> str:
    print(f"Habla ahora durante {duration} segundos...")
    recording = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="float32",
    )
    sd.wait()
    sf.write(output_path, recording, sample_rate)
    print(f"Audio guardado en: {output_path}")
    return output_path


whisper_transcriber = WhisperTranscriber(model_size="base")
piper_synthesizer = PiperSynthesizer(
    voice_name="es_ES-davefx-medium",
    data_dir="../models/piper",
)
voice_agent = VoiceAgent(
    transcriber=whisper_transcriber,
    llm=OllamaClient(model="qwen2.5:3b"),
    synthesizer=piper_synthesizer,
)

print("Agente de voz preparado.")

WhisperTranscriber preparado con faster-whisper.


## 14. Sintesis local con Piper

Piper sustituye ElevenLabs y genera un archivo WAV a partir del texto. Instala el ejecutable y descarga una voz antes de ejecutar esta celda.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
from IPython.display import Audio, display


SAMPLE_RATE = 16_000
AUDIO_DIR = Path("../models/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)


def create_voice_button(include_audio: bool = True) -> widgets.VBox:
    output = widgets.Output()
    record_button = widgets.Button(
        description="Hablar",
        button_style="primary",
        icon="microphone",
    )
    status = widgets.Label(value="Pulsa para comenzar a hablar")
    recording_state = {"stream": None, "frames": []}

    def audio_callback(indata, _frames, _time_info, callback_status) -> None:
        if callback_status:
            with output:
                print(f"Aviso de audio: {callback_status}")
        recording_state["frames"].append(indata.copy())

    def start_recording() -> None:
        recording_state["frames"] = []
        recording_state["stream"] = sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            callback=audio_callback,
        )
        recording_state["stream"].start()
        record_button.description = "Terminar y responder"
        record_button.button_style = "warning"
        status.value = "Grabando... pulsa otra vez para terminar"

    def stop_recording_and_respond() -> None:
        stream = recording_state["stream"]
        stream.stop()
        stream.close()
        recording_state["stream"] = None
        record_button.disabled = True
        record_button.description = "Procesando..."
        status.value = "Transcribiendo..."

        try:
            if not recording_state["frames"]:
                raise RuntimeError("No se ha capturado audio.")

            recording = np.concatenate(recording_state["frames"], axis=0)
            user_audio = AUDIO_DIR / "usuario_boton.wav"
            sf.write(user_audio, recording, SAMPLE_RATE)
            print(f"Audio guardado en: {user_audio}")

            user_text = voice_agent.transcriber.transcribe(str(user_audio))
            if not user_text:
                print("No se ha detectado texto. Inténtalo de nuevo.")
                status.value = "No se detectó voz"
                return

            print(f"Tú: {user_text}")
            voice_agent.messages.append({"role": "user", "content": user_text})
            status.value = "Generando respuesta..."
            answer = voice_agent.llm.chat(voice_agent.messages)
            voice_agent.messages.append({"role": "assistant", "content": answer})
            print(f"Agente: {answer}")

            if include_audio:
                response_audio = voice_agent.synthesizer.synthesize(
                    answer,
                    str(AUDIO_DIR / "respuesta_boton.wav"),
                )
                audio_data, audio_rate = sf.read(response_audio, dtype="float32")
                sd.play(audio_data, samplerate=audio_rate)
                sd.wait()
                print("Respuesta reproducida por los altavoces.")
                display(Audio(filename=response_audio, autoplay=False))

            status.value = "Respuesta preparada"
        except Exception as error:
            status.value = "Error"
            print(f"Error durante la interacción: {type(error).__name__}: {error}")
        finally:
            record_button.disabled = False
            record_button.description = "Hablar"
            record_button.button_style = "primary"

    def on_record_clicked(_button) -> None:
        with output:
            output.clear_output()
            if recording_state["stream"] is None:
                try:
                    start_recording()
                except Exception as error:
                    status.value = "Error al abrir el micrófono"
                    print(f"Error de micrófono: {type(error).__name__}: {error}")
            else:
                stop_recording_and_respond()

    record_button.on_click(on_record_clicked)
    return widgets.VBox([status, record_button, output])


voice_button = create_voice_button(include_audio=True)
display(voice_button)

PiperSynthesizer preparado con la voz es_ES-davefx-medium.


## 15. Orquestar el agente de voz por turnos

`VoiceAgent` coordina la transcripción, el historial, la respuesta de Ollama y la síntesis. El VAD y la captura del micrófono se incorporan alrededor de `process_turn`.

In [27]:
class VoiceAgent:
    def __init__(self, transcriber, llm, synthesizer) -> None:
        self.transcriber = transcriber
        self.llm = llm
        self.synthesizer = synthesizer
        self.messages = [
            {
                "role": "system",
                "content": "Eres un asistente de voz útil. Responde en español y de forma concisa.",
            }
        ]

    def process_turn(self, audio_path: str, output_path: str) -> str:
        user_text = self.transcriber.transcribe(audio_path)
        if not user_text:
            return ""

        self.messages.append({"role": "user", "content": user_text})
        answer = self.llm.chat(self.messages)
        self.messages.append({"role": "assistant", "content": answer})
        self.synthesizer.synthesize(answer, output_path)
        return answer

print("VoiceAgent definido: audio -> texto -> Ollama -> audio.")

VoiceAgent definido: audio -> texto -> Ollama -> audio.


## 16. Interactuar con el agente mediante el micrófono

Esta sección crea una interacción local por turnos, similar al flujo de `Version_online_agente_voz.ipynb`:

```text
Micrófono -> faster-whisper -> Ollama -> Piper -> respuesta hablada
```

Para evitar que el micrófono quede abierto indefinidamente, cada turno se graba durante un número de segundos configurado. Después se puede repetir el turno.

In [28]:
from pathlib import Path

import sounddevice as sd
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE = 16_000
RECORD_SECONDS = 5
AUDIO_DIR = Path("models/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)


def record_microphone(
    output_path: str = str(AUDIO_DIR / "usuario.wav"),
    duration: int = RECORD_SECONDS,
    sample_rate: int = SAMPLE_RATE,
) -> str:
    print(f"Habla ahora durante {duration} segundos...")
    recording = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype="float32",
    )
    sd.wait()
    sf.write(output_path, recording, sample_rate)
    print(f"Audio guardado en: {output_path}")
    return output_path


whisper_transcriber = WhisperTranscriber(model_size="base")
piper_synthesizer = PiperSynthesizer(
    voice_name="es_ES-davefx-medium",
    data_dir="models/piper",
)
voice_agent = VoiceAgent(
    transcriber=whisper_transcriber,
    llm=OllamaClient(model="qwen2.5:3b"),
    synthesizer=piper_synthesizer,
)

print("Agente de voz preparado.")

Agente de voz preparado.


## 17. Comprobar el micrófono disponible

Ejecuta esta celda para ver los dispositivos de audio. El agente utilizará el dispositivo de entrada predeterminado de Windows.

In [29]:
import sounddevice as sd

print("Dispositivo de entrada predeterminado:")
print(sd.query_devices(kind="input"))
print("\nTodos los dispositivos:")
print(sd.query_devices())

Dispositivo de entrada predeterminado:
{'name': 'Varios micrófonos (Realtek(R) A', 'index': 1, 'hostapi': 0, 'max_input_channels': 2, 'max_output_channels': 0, 'default_low_input_latency': 0.09, 'default_low_output_latency': 0.09, 'default_high_input_latency': 0.18, 'default_high_output_latency': 0.18, 'default_samplerate': 44100.0}

Todos los dispositivos:
   0 Asignador de sonido Microsoft - Input, MME (2 in, 0 out)
>  1 Varios micrófonos (Realtek(R) A, MME (2 in, 0 out)
   2 Asignador de sonido Microsoft - Output, MME (0 in, 2 out)
<  3 Altavoces (Realtek(R) Audio), MME (0 in, 2 out)
   4 Controlador primario de captura de sonido, Windows DirectSound (2 in, 0 out)
   5 Varios micrófonos (Realtek(R) Audio), Windows DirectSound (2 in, 0 out)
   6 Controlador primario de sonido, Windows DirectSound (0 in, 2 out)
   7 Altavoces (Realtek(R) Audio), Windows DirectSound (0 in, 2 out)
   8 Altavoces (Realtek(R) Audio), Windows WASAPI (0 in, 2 out)
   9 Varios micrófonos (Realtek(R) Audio), 

## 18. Hablar, ver la respuesta y escucharla

Ejecuta la celda siguiente. Pulsa **Hablar** para comenzar y **Terminar y responder** cuando acabes. El agente mostrará la transcripción, la respuesta escrita y reproducirá la respuesta con Piper.

In [30]:
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import sounddevice as sd
import soundfile as sf
from IPython.display import Audio, display


SAMPLE_RATE = 16_000
AUDIO_DIR = Path("models/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)


def create_voice_button(include_audio: bool = True) -> widgets.VBox:
    output = widgets.Output()
    record_button = widgets.Button(
        description="Hablar",
        button_style="primary",
        icon="microphone",
    )
    status = widgets.Label(value="Pulsa para comenzar a hablar")
    recording_state = {"stream": None, "frames": []}

    def audio_callback(indata, _frames, _time_info, callback_status) -> None:
        if callback_status:
            with output:
                print(f"Aviso de audio: {callback_status}")
        recording_state["frames"].append(indata.copy())

    def start_recording() -> None:
        recording_state["frames"] = []
        recording_state["stream"] = sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="float32",
            callback=audio_callback,
        )
        recording_state["stream"].start()
        record_button.description = "Terminar y responder"
        record_button.button_style = "warning"
        status.value = "Grabando... pulsa otra vez para terminar"

    def stop_recording_and_respond() -> None:
        stream = recording_state["stream"]
        stream.stop()
        stream.close()
        recording_state["stream"] = None
        record_button.disabled = True
        record_button.description = "Procesando..."
        status.value = "Transcribiendo..."

        try:
            if not recording_state["frames"]:
                raise RuntimeError("No se ha capturado audio.")

            recording = np.concatenate(recording_state["frames"], axis=0)
            user_audio = AUDIO_DIR / "usuario_boton.wav"
            sf.write(user_audio, recording, SAMPLE_RATE)
            print(f"Audio guardado en: {user_audio}")

            user_text = voice_agent.transcriber.transcribe(str(user_audio))
            if not user_text:
                print("No se ha detectado texto. Inténtalo de nuevo.")
                status.value = "No se detectó voz"
                return

            print(f"Tú: {user_text}")
            voice_agent.messages.append({"role": "user", "content": user_text})
            status.value = "Generando respuesta..."
            answer = voice_agent.llm.chat(voice_agent.messages)
            voice_agent.messages.append({"role": "assistant", "content": answer})
            print(f"Agente: {answer}")

            if include_audio:
                response_audio = voice_agent.synthesizer.synthesize(
                    answer,
                    str(AUDIO_DIR / "respuesta_boton.wav"),
                )
                audio_data, audio_rate = sf.read(response_audio, dtype="float32")
                sd.play(audio_data, samplerate=audio_rate)
                sd.wait()
                print("Respuesta reproducida por los altavoces.")
                display(Audio(filename=response_audio, autoplay=False))

            status.value = "Respuesta preparada"
        except Exception as error:
            status.value = "Error"
            print(f"Error durante la interacción: {type(error).__name__}: {error}")
        finally:
            record_button.disabled = False
            record_button.description = "Hablar"
            record_button.button_style = "primary"

    def on_record_clicked(_button) -> None:
        with output:
            output.clear_output()
            if recording_state["stream"] is None:
                try:
                    start_recording()
                except Exception as error:
                    status.value = "Error al abrir el micrófono"
                    print(f"Error de micrófono: {type(error).__name__}: {error}")
            else:
                stop_recording_and_respond()

    record_button.on_click(on_record_clicked)
    return widgets.VBox([status, record_button, output])


voice_button = create_voice_button(include_audio=True)
display(voice_button)

## 19. Hablar y recibir solo texto

Esta versión utiliza el mismo botón manual de la sección 18, pero no genera ni reproduce audio. Pulsa **Hablar** para comenzar y **Terminar y responder** cuando acabes; se mostrarán la transcripción y la respuesta de Ollama.

In [31]:
voice_button_text_only = create_voice_button(include_audio=False)
display(voice_button_text_only)

## 20. Probar el agente completamente en local

Antes de utilizar el micrófono, esta prueba usa el audio `models/audio/prueba_kernel.wav` ya creado. Comprueba el circuito completo sin entrada en tiempo real:

```text
WAV -> faster-whisper -> Ollama -> Piper -> altavoces
```

In [ ]:
from pathlib import Path

import sounddevice as sd
import soundfile as sf

TEST_AUDIO = Path("../models/audio/prueba_kernel.wav").resolve()
TEST_RESPONSE = Path("../models/audio/respuesta_local.wav").resolve()

if not TEST_AUDIO.exists():
    raise FileNotFoundError(f"No existe el audio de prueba: {TEST_AUDIO}")

print(f"Transcribiendo: {TEST_AUDIO}")
local_transcriber = WhisperTranscriber(model_size="base")
transcribed_text = local_transcriber.transcribe(str(TEST_AUDIO))
print(f"Texto transcrito: {transcribed_text or '(vacío)'}")

if transcribed_text:
    local_messages = [
        {
            "role": "system",
            "content": "Eres un asistente de voz útil. Responde en español y de forma concisa.",
        },
        {"role": "user", "content": transcribed_text},
    ]
    local_answer = OllamaClient(model="qwen2.5:3b").chat(local_messages)
    print(f"Respuesta de Ollama: {local_answer}")

    local_synthesizer = PiperSynthesizer(data_dir="../models/piper")
    local_synthesizer.synthesize(local_answer, str(TEST_RESPONSE))
    audio_data, audio_rate = sf.read(str(TEST_RESPONSE), dtype="float32")
    print(f"Audio generado: {TEST_RESPONSE}")
    sd.play(audio_data, samplerate=audio_rate)
    sd.wait()
    print("Prueba local terminada.")
else:
    print("No se ha detectado texto en el audio de prueba.")

Transcribiendo: C:\Users\oitav\Documents\VIU\TFM\curso tfm\models\audio\prueba_kernel.wav
Texto transcrito: Hola, esta es una prueba de voz.
Respuesta de Ollama: ¡Hola! ¿En qué puedo ayudarte hoy?
Audio generado: C:\Users\oitav\Documents\VIU\TFM\curso tfm\models\audio\respuesta_local.wav
Prueba local terminada.
